# Lesson 02 - Prozkoumávání Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** je sjednocený rámec pro vytváření AI agentů. Poskytuje čistou, kompozitní architekturu se čtyřmi základními stavebními kameny:

- **Client** – připojuje se k AI modelovému koncovému bodu a zajišťuje komunikaci
- **Agent** – obaluje klienta s instrukcemi a definicemi nástrojů
- **Tools** – rozšiřují schopnosti agenta pomocí vlastních funkcí, které může model volat
- **Session** – udržuje historii konverzace pro vícetahové interakce

V této lekci vytvoříme **agenta pro rezervaci cestování**, který kontroluje dostupnost destinace pomocí těchto konceptů.


## Nastavení


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Porozumění architektuře Agent Frameworku

Microsoft Agent Framework následuje vrstvenou architekturu:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` se připojuje k nasazení Azure OpenAI. Zajišťuje autentizaci, formátování požadavků a parsování odpovědí.
2. **Agent** – Vytvořený z klienta pomocí `provider.create_agent()`, agent kombinuje přístup k modelu s instrukcemi (systémový prompt) a nástroji.
3. **Tools** – Python funkce dekorované `@tool`, které může agent volat k provádění akcí nebo získávání dat.
4. **Session** – Objekt `AgentSession` (vytvořený přes `agent.create_session()`), který uchovává historii konverzace, což umožňuje vícenásobný dialog, kde si agent pamatuje předchozí kontext.

Pojďme si vybudovat každou vrstvu krok po kroku.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Přidání nástrojů s dekorátorem @tool

Nástroje umožňují agentům provádět akce přesahující generování textu. Dekorátor `@tool` převede běžnou Python funkci na něco, co může agent volat.

Hlavní body:
- Použijte `Annotated[type, "popis"]`, aby model rozuměl každému parametru.
- Docstring se stane popisem nástroje, který model uvidí.
- `approval_mode="never_require"` znamená, že nástroj běží automaticky bez potvrzení uživatele.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Vytvoření agenta s nástroji

Nyní kombinujeme klienta, instrukce a nástroje do agenta. `instructions` slouží jako systémový prompt — definují osobnost a chování agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Vícekolové konverzace se sezeními

`AgentSession` (vytvořené pomocí `agent.create_session()`) sleduje všechny zprávy v konverzaci. Předáváním stejného sezení každému volání `agent.run()` má agent přístup k celé historii konverzace a může se odkazovat na dřívější zprávy.

Předáváme `tools=[check_destination_availability]`, aby agent mohl během každého kola volat náš kontrolor dostupnosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Shrnutí

V této lekci jste prozkoumali čtyři pilíře Microsoft Agent Frameworku:

| Koncept | Co jste se naučili |
|---------|--------------------|
| **Klient** | `AzureAIProjectAgentProvider` se připojuje k Azure OpenAI s autentizací založenou na přihlašovacích údajích |
| **Agent** | `provider.create_agent()` spojuje připojení k modelu s instrukcemi a názvem |
| **Nástroje** | Dekorátor `@tool` zpřístupňuje Python funkce, které může agent volat |
| **Sezení** | `agent.create_session()` uchovává historii konverzace přes více kol |

Tyto stavební bloky se skládají dohromady a vytvářejí agenty, kteří dokážou vést přirozené konverzace, volat externí funkce a udržovat kontext — základ pro pokročilejší agentní vzory v následujících lekcích.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho rodném jazyce by měl být považován za autoritativní zdroj. Pro důležité informace se doporučuje profesionální lidský překlad. Nejsme odpovědni za jakékoli nedorozumění nebo mylné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
